# Python EDA - NHAMCS 2022 Emergency Department Visits

This notebook starts the Python exploratory analysis for the ED patient flow project. It validates the prepared dataset, checks missingness and duplicates, explores arrival volume patterns, compares 2-hour and 4-hour wait targets, and records early ED operations insights for the future dashboard and report.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path('..')
DATA_PATH = ROOT / 'data' / 'processed' / 'nhamcs_2022_visits_clean.csv'
FIGURE_DIR = ROOT / 'outputs' / 'figures'

df = pd.read_csv(DATA_PATH, na_values=['NULL'])
df.shape

(16025, 39)

## Dataset Validation

Expected validation totals from the project brief:

- 16,025 ED visits
- 39 selected analysis fields in the clean SQL handoff file
- 13,272 valid wait times
- 238 visits over the 4-hour target
- 907 visits over the 2-hour target

In [2]:
validation = {
    'rows': len(df),
    'columns': df.shape[1],
    'duplicate_visit_ids': int(df['visit_id'].duplicated().sum()),
    'valid_wait_times': int(df['wait_time_minutes'].notna().sum()),
    'missing_wait_times': int(df['wait_time_minutes'].isna().sum()),
    'four_hour_waits': int(df['long_wait_4hr_flag'].sum()),
    'two_hour_waits': int(df['extended_wait_2hr_flag'].sum()),
}
validation

{'rows': 16025,
 'columns': 39,
 'duplicate_visit_ids': 0,
 'valid_wait_times': 13272,
 'missing_wait_times': 2753,
 'four_hour_waits': 238,
 'two_hour_waits': 907}

In [3]:
df.dtypes.to_frame('dtype')

,dtype
visit_id,int64
visit_month,int64
visit_month_name,str
visit_day,str
arrival_time,str
arrival_hour,float64
wait_time_minutes,float64
visit_length_minutes,float64
age_years,int64
pain_scale,float64


In [4]:
missing_summary = (
    df.isna().sum()
    .to_frame('missing_rows')
    .assign(missing_rate=lambda x: (x['missing_rows'] / len(df) * 100).round(1))
    .sort_values('missing_rows', ascending=False)
)
missing_summary.head(15)

,missing_rows,missing_rate
pain_scale,7055,44.0
extended_wait_2hr_flag,2753,17.2
wait_time_minutes,2753,17.2
long_wait_4hr_flag,2753,17.2
diastolic_bp,1754,10.9
systolic_bp,1722,10.7
pulse_bpm,1053,6.6
oxygen_saturation,969,6.0
temperature_f,937,5.8
respiratory_rate,856,5.3


## ED Visit Volume

Early volume charts were exported to `outputs/figures/` for review and later dashboard selection:

- `ed_visits_by_month.png`
- `ed_visits_by_day.png`
- `ed_visits_by_hour.png`

In [5]:
month_order = list(range(1, 13))
month_volume = df.groupby(['visit_month', 'visit_month_name']).size().rename('visits').reset_index()
month_volume.sort_values('visit_month')

,visit_month,visit_month_name,visits
0,1,January,1468
1,2,February,1619
2,3,March,1388
3,4,April,1367
4,5,May,1135
5,6,June,1408
6,7,July,1448
7,8,August,1293
8,9,September,763
9,10,October,1261


In [6]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_volume = df.groupby('visit_day').size().reindex(day_order).rename('visits')
day_volume

visit_day
Monday       2631
Tuesday      2415
Wednesday    2374
Thursday     2301
Friday       2259
Saturday     2027
Sunday       2018
Name: visits, dtype: int64

In [7]:
hour_volume = df.groupby('arrival_hour').size().reindex(range(24), fill_value=0).rename('visits')
hour_volume.sort_values(ascending=False).head(10)

arrival_hour
10    1017
13    1004
12     971
11     960
14     902
17     894
16     878
18     877
19     841
15     839
Name: visits, dtype: int64

## Wait-Time Analysis

The 4-hour target identifies a very small high-wait group, while the 2-hour target captures more cases and is more practical as the first ML classification target.

In [8]:
wait_summary = df['wait_time_minutes'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
wait_targets = pd.Series({
    'valid_wait_times': df['wait_time_minutes'].notna().sum(),
    'mean_wait_minutes': round(df['wait_time_minutes'].mean(), 1),
    'median_wait_minutes': round(df['wait_time_minutes'].median(), 1),
    'two_hour_wait_count': int(df['extended_wait_2hr_flag'].sum()),
    'two_hour_wait_rate_valid_waits': round(df['extended_wait_2hr_flag'].sum() / df['wait_time_minutes'].notna().sum() * 100, 2),
    'four_hour_wait_count': int(df['long_wait_4hr_flag'].sum()),
    'four_hour_wait_rate_valid_waits': round(df['long_wait_4hr_flag'].sum() / df['wait_time_minutes'].notna().sum() * 100, 2),
})
wait_summary, wait_targets

(count    13272.000000
 mean        36.026447
 std         65.500736
 min          0.000000
 25%          5.000000
 50%         14.000000
 75%         39.000000
 90%         91.000000
 95%        146.000000
 99%        301.000000
 max       1280.000000
 Name: wait_time_minutes, dtype: float64,
 valid_wait_times                   13272.00
 mean_wait_minutes                     36.00
 median_wait_minutes                   14.00
 two_hour_wait_count                  907.00
 two_hour_wait_rate_valid_waits         6.83
 four_hour_wait_count                 238.00
 four_hour_wait_rate_valid_waits        1.79
 dtype: float64)

In [9]:
def wait_by_group(column):
    return (
        df.groupby(column, dropna=False)['wait_time_minutes']
        .agg(valid_waits='count', mean_wait='mean', median_wait='median')
        .round(1)
        .sort_values('median_wait', ascending=False)
    )

wait_by_group('triage_level')

,valid_waits,mean_wait,median_wait
triage_level,,,
Facility does not conduct triage,916,35.9,19.0
No triage,426,45.3,18.0
Urgent,4525,37.6,15.0
Semi-urgent,2384,35.5,15.0
Nonurgent,263,30.8,13.0
Unknown,3292,37.1,13.0
Emergent,1354,28.5,12.0
Immediate,112,24.9,9.5


In [10]:
for column in ['arrival_by_ambulance', 'sex', 'region', 'metropolitan_status']:
    display(wait_by_group(column))

,valid_waits,mean_wait,median_wait
arrival_by_ambulance,,,
No,10607,37.6,16.0
Unknown,274,38.6,16.0
Yes,2391,28.7,10.0


,valid_waits,mean_wait,median_wait
sex,,,
Female,7065,36.9,15.0
Male,6207,35.1,14.0


,valid_waits,mean_wait,median_wait
region,,,
South,3713,41.2,20.0
Northeast,2624,43.3,17.5
West,3066,27.7,12.0
Midwest,3869,32.7,10.0


,valid_waits,mean_wait,median_wait
metropolitan_status,,,
Metropolitan,11747,36.8,15.0
Non-metropolitan,1525,29.8,12.0


In [11]:
age_bins = [-1, 17, 34, 49, 64, 200]
age_labels = ['0-17', '18-34', '35-49', '50-64', '65+']
df['age_group'] = pd.cut(df['age_years'], bins=age_bins, labels=age_labels)
wait_by_group('age_group')

,valid_waits,mean_wait,median_wait
age_group,,,
0-17,2795,39.3,17.0
35-49,2228,37.5,16.0
18-34,3107,34.9,14.0
50-64,2388,35.2,13.0
65+,2754,33.5,12.0


## Outcomes and Operational Pressure

Outcome fields are useful for ED operational reporting but should not be used as arrival-time prediction features because they happen after arrival.

In [12]:
outcome_columns = [
    'admitted_to_hospital',
    'observation_then_hospitalized',
    'observation_then_discharged',
    'left_without_being_seen',
    'left_before_treatment_complete',
    'left_against_medical_advice',
    'died_in_ed',
]

outcome_summary = pd.DataFrame({
    'yes_count': [(df[col] == 'Yes').sum() for col in outcome_columns],
    'yes_rate': [round((df[col] == 'Yes').mean() * 100, 2) for col in outcome_columns],
}, index=outcome_columns)
outcome_summary

,yes_count,yes_rate
admitted_to_hospital,2121,13.24
observation_then_hospitalized,177,1.10
observation_then_discharged,214,1.34
left_without_being_seen,327,2.04
left_before_treatment_complete,221,1.38
left_against_medical_advice,228,1.42
died_in_ed,33,0.21


## Early Findings

- Python validates the expected 16,025 rows, 13,272 valid wait times, 238 four-hour waits, and 907 two-hour waits.
- Median wait time is 14 minutes and average wait time is 36.0 minutes, which suggests the distribution is right-skewed by a smaller group of long waits.
- February is the highest-volume month in this prepared sample, Monday is the busiest arrival day, and 10:00 is the busiest arrival hour.
- The 2-hour target flags 6.83% of visits with valid wait times, compared with 1.79% for the 4-hour target. The 2-hour target is the better first ML target because it is less imbalanced.
- Ambulance arrivals have a lower median wait than non-ambulance arrivals, which is consistent with priority triage and should be explained carefully in the final report.
- Region and metropolitan status show visible wait variation and should be kept for ED pressure analysis.
- Post-arrival outcomes, including admission, observation, visit length, and leaving-before-care fields, should be excluded from arrival-time prediction features.

## Dashboard Visual Candidates

First dashboard candidates:

- KPI cards: total ED visits, median wait, average wait, 2-hour extended-wait rate, 4-hour long-wait rate, admission rate, LWBS rate, ambulance-arrival rate.
- Charts: visits by month, visits by day, visits by arrival hour, wait distribution with thresholds, median wait by triage level, median wait by region, outcome-rate summary.
- Filters: month, day, arrival hour or arrival period, triage level, ambulance arrival, age group, sex, region, and metropolitan status.